# 🧠 PersonaPlex 7B → Slim — NVIDIA Minitron-Style Pipeline
### V8 Production | Depth + Width Pruning | Classical KD | Live Monitoring

---
## Strategy (NVIDIA Minitron Blog verified)
| Step | Method | Target |
|------|--------|--------|
| 1 | Activation-based importance scoring (forward-only) | All axes |
| 2 | **Depth pruning** — remove least important layers | 32 → 24 layers |
| 3 | **Width pruning** — heads + FFN neurons | 25% heads, 30% FFN |
| 4 | Classical KD — logit + hidden distillation | 5 epochs |

## Key improvements over V7
- ✅ Depth + Width pruning (Minitron approach)
- ✅ Activation-only importance (no backward pass needed)
- ✅ 8-bit AdamW optimizer (VRAM efficient)
- ✅ Teacher CPU offload during optimizer step
- ✅ Live loss graph + tokens/sec + VRAM monitor
- ✅ Auto-resume from Drive checkpoint
- ✅ Auto-delete old checkpoints

## Requirements
- A100 80GB High-RAM runtime
- HF_TOKEN secret set (nvidia/personaplex-7b-v1 access)
- Google Drive mounted


## 📦 Section 0: Install
⚠️ Restart runtime after this cell.

In [ ]:
!pip install -q "transformers>=4.45.0" "accelerate>=0.34.0"
!pip install -q "datasets>=2.20.0" "huggingface_hub>=0.26.0"
!pip install -q "safetensors" "sentencepiece" "bitsandbytes>=0.43.0"
!pip install -q "tqdm" "matplotlib" "seaborn" "psutil" "ipywidgets"

import importlib
print("Versions:")
for pkg in ['transformers','accelerate','bitsandbytes','torch']:
    print(f"  {pkg}: {importlib.import_module(pkg).__version__}")
print("\n✅ Done — restart runtime now")


## 🔐 Section 1: Auth + Config + Drive

In [ ]:
import os, gc, copy, math, time, json, shutil, glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import psutil
from datetime import datetime

# HuggingFace auth
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    assert HF_TOKEN
    print("✅ HF Token from Secrets")
except:
    HF_TOKEN = "hf_xxxx"  # fallback
    print("⚠️  Manual token — update this!")

from huggingface_hub import login
login(token=HF_TOKEN)
os.environ['HF_TOKEN'] = HF_TOKEN

# Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/PersonaPlex_V8'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    DRIVE_OK = True
    print(f"✅ Drive mounted → {DRIVE_DIR}")
except Exception as e:
    DRIVE_DIR = '/content/personaplex_v8'
    DRIVE_OK = False
    print(f"⚠️  Drive failed: {e}")

# VRAM fragmentation fix
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

CONFIG = {
    "model_id"           : "nvidia/personaplex-7b-v1",
    "local_dir"          : "/content/personaplex_v8",
    "drive_dir"          : DRIVE_DIR,

    # Depth pruning — NVIDIA: remove middle layers, keep first+last
    "depth_pruning_ratio": 0.25,      # 25% layers removed: 32 → 24 layers

    # Width pruning
    "head_pruning_ratio" : 0.25,      # 25% heads removed per layer
    "ffn_pruning_ratio"  : 0.30,      # 30% FFN neurons removed per layer

    # Distillation — NVIDIA Minitron verified params
    "distill_epochs"     : 5,
    "distill_lr"         : 1e-4,      # NVIDIA: peak lr=1e-4
    "min_lr"             : 1e-5,      # NVIDIA: min lr=1e-5
    "warmup_steps"       : 40,        # NVIDIA: linear warmup 40 steps
    "batch_size"         : 1,
    "grad_accum_steps"   : 32,        # Effective batch = 32
    "max_seq_len"        : 512,
    "temperature"        : 2.0,

    # Loss weights — logit-only style (width pruning, depth not extreme)
    "alpha_ce"           : 0.7,
    "alpha_kd"           : 0.2,
    "alpha_hidden"       : 0.1,

    # Audio
    "num_audio_codebooks": 8,
    "audio_vocab_size"   : 2048,

    # Checkpoint
    "save_every_n_epochs": 1,
    "keep_last_n"        : 1,

    "dtype"  : torch.bfloat16,
    "device" : "cuda" if torch.cuda.is_available() else "cpu",
}

os.makedirs(CONFIG['local_dir'], exist_ok=True)
os.makedirs(CONFIG['drive_dir'], exist_ok=True)

# System info
print(f"\n{'='*55}")
print(f"  SYSTEM INFO")
print(f"{'='*55}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"  GPU   : {gpu.name}")
    print(f"  VRAM  : {gpu.total_memory/1e9:.1f} GB")
ram = psutil.virtual_memory()
print(f"  RAM   : {ram.total/1e9:.1f} GB")
disk = shutil.disk_usage('/content')
print(f"  Disk  : {disk.free/1e9:.1f} GB free")
print(f"  Drive : {'✅' if DRIVE_OK else '❌'}")
print(f"{'='*55}")
print(f"  Depth prune : {CONFIG['depth_pruning_ratio']*100:.0f}% layers")
print(f"  Head prune  : {CONFIG['head_pruning_ratio']*100:.0f}%")
print(f"  FFN prune   : {CONFIG['ffn_pruning_ratio']*100:.0f}%")
print(f"  Epochs      : {CONFIG['distill_epochs']}")
print(f"  LR          : {CONFIG['distill_lr']} → {CONFIG['min_lr']}")
print(f"{'='*55}")


## 📥 Section 2: Load Teacher

In [ ]:
from transformers import MoshiForConditionalGeneration, AutoTokenizer

print("🔄 Loading PersonaPlex 7B...")
t0 = time.time()

teacher_lm = MoshiForConditionalGeneration.from_pretrained(
    CONFIG['model_id'],
    torch_dtype=CONFIG['dtype'],
    device_map="auto",
    token=HF_TOKEN,
)
teacher_lm.eval()
for p in teacher_lm.parameters():
    p.requires_grad_(False)

# Tokenizer
try:
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_id'], token=HF_TOKEN)
    print(f"✅ Tokenizer: {type(tokenizer).__name__}")
except Exception as e:
    tokenizer = None
    print(f"⚠️  Tokenizer failed: {e}")

t_p = sum(p.numel() for p in teacher_lm.parameters()) / 1e9
vram = torch.cuda.memory_allocated() / 1e9
print(f"✅ Teacher loaded in {time.time()-t0:.0f}s")
print(f"   Params : {t_p:.2f}B")
print(f"   VRAM   : {vram:.1f} GB")


## 🔍 Section 3: Architecture Verify

In [ ]:
def find_temporal_transformer(model):
    candidates = [
        lambda m: (getattr(getattr(m,'decoder',None),'model',None), 'decoder.model'),
        lambda m: (getattr(m,'decoder',None), 'decoder'),
    ]
    for fn in candidates:
        c, name = fn(model)
        if c is None: continue
        if getattr(c,'layers',None) is not None and len(c.layers) > 0:
            return c, name
        inner = getattr(c,'model',None)
        if inner is not None and getattr(inner,'layers',None) is not None:
            return inner, f"{name}.model"
    return None, None

temporal_tf, tf_path = find_temporal_transformer(teacher_lm)
assert temporal_tf is not None, "Temporal transformer not found!"

layer0 = temporal_tf.layers[0]
attn0  = layer0.self_attn
mlp0   = layer0.mlp

ARCH = {
    'num_layers' : len(temporal_tf.layers),
    'num_heads'  : attn0.num_heads,
    'num_kv'     : attn0.num_key_value_heads,
    'head_dim'   : attn0.head_dim,
    'embed_dim'  : attn0.hidden_size,
    'ffn_interm' : mlp0.fc1.weight.shape[0],
    'ffn_half'   : mlp0.fc1.weight.shape[0] // 2,
}

print(f"✅ Found at: '{tf_path}'")
print(f"   Layers     : {ARCH['num_layers']}")
print(f"   Embed dim  : {ARCH['embed_dim']}")
print(f"   Heads      : {ARCH['num_heads']} (head_dim={ARCH['head_dim']})")
print(f"   FFN        : {ARCH['ffn_interm']} (half={ARCH['ffn_half']})")


## 📊 Section 4: Importance Scoring (NVIDIA Minitron Method)

**Activation-only** — no backward pass needed (faster + less VRAM).
- **Depth**: Drop each layer, measure LM loss increase → most important layers = highest loss increase
- **Width (heads)**: Activation magnitude on attention output
- **Width (FFN)**: Activation magnitude on fc1 output


In [ ]:
import numpy as np
from tqdm.notebook import tqdm

# ── Calibration data — 1024 samples (NVIDIA uses 1024) ──
print("🔄 Building calibration data (1024 samples)...")
calib_ids = [torch.randint(100, 32000, (64,)) for _ in range(1024)]
print(f"✅ {len(calib_ids)} calibration samples ready")

def get_lm_loss(model, sample_ids, device, n=32):
    "Quick LM loss estimate on n samples"
    model.eval()
    total = 0.0
    with torch.no_grad():
        for ids in sample_ids[:n]:
            ids = ids.unsqueeze(0).to(device)
            aud = torch.randint(0, CONFIG['audio_vocab_size'],
                                (1, CONFIG['num_audio_codebooks'], ids.shape[1]),
                                dtype=torch.long, device=device)
            try:
                out = model(input_ids=ids, moshi_audio_codes=aud, user_audio_codes=aud)
                lg = out.logits[:, :-1].float()
                l = F.cross_entropy(lg.reshape(-1, lg.size(-1)), ids[:, 1:].reshape(-1))
                total += l.item()
            except:
                pass
    return total / n

# ══════════════════════════════════════════
# 4.1 DEPTH IMPORTANCE — Drop layer, measure loss
# ══════════════════════════════════════════
print("\n📊 Depth importance scoring...")
print("   (Dropping each layer temporarily and measuring LM loss)")

baseline_loss = get_lm_loss(teacher_lm, calib_ids, CONFIG['device'])
print(f"   Baseline LM loss: {baseline_loss:.4f}")

layer_importance = {}
NL = ARCH['num_layers']

for l in tqdm(range(NL), desc="Layer importance"):
    # Temporarily remove layer
    orig_layer = temporal_tf.layers[l]
    temporal_tf.layers[l] = nn.Identity()
    try:
        loss = get_lm_loss(teacher_lm, calib_ids, CONFIG['device'], n=16)
        # Importance = how much loss increases when layer removed
        layer_importance[l] = loss - baseline_loss
    except:
        layer_importance[l] = 0.0
    temporal_tf.layers[l] = orig_layer
    torch.cuda.empty_cache()

print(f"✅ Depth scores computed")
for l in range(NL):
    print(f"   Layer {l:2d}: importance={layer_importance[l]:.4f}", end="")
    if l % 4 == 3: print()

# ══════════════════════════════════════════
# 4.2 WIDTH — HEAD IMPORTANCE (Activation magnitude)
# ══════════════════════════════════════════
print("\n📊 Head importance scoring (activation magnitude)...")

head_scores = {l: [] for l in range(NL)}
hooks = []

def make_head_hook(l):
    def hook(mod, inp, out):
        # out: [B, T, NH*HD] → reshape → mean over B,T → [NH]
        NH = ARCH['num_heads']
        HD = ARCH['head_dim']
        o = out.detach().float().view(-1, NH, HD)
        head_scores[l].append(o.abs().mean(dim=(0, 2)).cpu())
    return hook

# Hook on attention output (o_proj input proxy = self_attn output)
for l, layer in enumerate(temporal_tf.layers):
    if hasattr(layer, 'self_attn'):
        hooks.append(layer.self_attn.register_forward_hook(make_head_hook(l)))

temporal_tf.eval()
with torch.no_grad():
    for ids in tqdm(calib_ids[:128], desc="Head scoring"):
        ids = ids.unsqueeze(0).to(CONFIG['device'])
        aud = torch.randint(0, CONFIG['audio_vocab_size'],
                            (1, CONFIG['num_audio_codebooks'], ids.shape[1]),
                            dtype=torch.long).to(CONFIG['device'])
        try:
            teacher_lm(input_ids=ids, moshi_audio_codes=aud, user_audio_codes=aud)
        except:
            pass

for h in hooks: h.remove()

head_scores_final = {}
for l in range(NL):
    if head_scores[l]:
        head_scores_final[l] = torch.stack(head_scores[l]).mean(0)
    else:
        head_scores_final[l] = torch.ones(ARCH['num_heads'])
print(f"✅ Head scores: {NL} layers × {ARCH['num_heads']} heads")

# ══════════════════════════════════════════
# 4.3 WIDTH — FFN IMPORTANCE (Activation magnitude)
# ══════════════════════════════════════════
print("\n📊 FFN importance scoring (activation magnitude)...")

ffn_scores = {l: [] for l in range(NL)}
hooks = []

def make_ffn_hook(l):
    def hook(mod, inp, out):
        ffn_scores[l].append(out.detach().float().abs().mean(dim=(0,1)).cpu())
    return hook

for l, layer in enumerate(temporal_tf.layers):
    if hasattr(layer, 'mlp'):
        hooks.append(layer.mlp.fc1.register_forward_hook(make_ffn_hook(l)))

with torch.no_grad():
    for ids in tqdm(calib_ids[:128], desc="FFN scoring"):
        ids = ids.unsqueeze(0).to(CONFIG['device'])
        aud = torch.randint(0, CONFIG['audio_vocab_size'],
                            (1, CONFIG['num_audio_codebooks'], ids.shape[1]),
                            dtype=torch.long).to(CONFIG['device'])
        try:
            teacher_lm(input_ids=ids, moshi_audio_codes=aud, user_audio_codes=aud)
        except:
            pass

for h in hooks: h.remove()

ffn_scores_final = {}
for l in range(NL):
    if ffn_scores[l]:
        ffn_scores_final[l] = torch.stack(ffn_scores[l]).mean(0)
    else:
        ffn_scores_final[l] = torch.ones(ARCH['ffn_interm'])
print(f"✅ FFN scores: {NL} layers × {ARCH['ffn_interm']} neurons")
torch.cuda.empty_cache()
gc.collect()


## ✂️ Section 5: Pruning — Depth + Width

**NVIDIA Minitron order:**
1. Depth prune — remove least important layers (keep first + last, remove middle)
2. Width prune — heads + FFN on remaining layers


In [ ]:
# ══════════════════════════════════════════
# 5.1 DEPTH PRUNING — Remove least important layers
# ══════════════════════════════════════════

# Which layers to REMOVE — lowest importance scores
# NVIDIA: first and last layers most important, middle ones prunable
n_remove = int(ARCH['num_layers'] * CONFIG['depth_pruning_ratio'])
n_keep   = ARCH['num_layers'] - n_remove

# Sort by importance — remove lowest scoring
sorted_layers = sorted(layer_importance.items(), key=lambda x: x[1])
remove_set    = set([l for l, _ in sorted_layers[:n_remove]])
keep_indices  = sorted([l for l in range(ARCH['num_layers']) if l not in remove_set])

print(f"Depth pruning: {ARCH['num_layers']} → {n_keep} layers")
print(f"Removing layers: {sorted(remove_set)}")
print(f"Keeping layers : {keep_indices}")

# ══════════════════════════════════════════
# 5.2 WIDTH PRUNING FUNCTIONS
# ══════════════════════════════════════════

def prune_attention(attn, arch, head_mask):
    "Prune MoshiSdpaAttention — MoshiLinear wrapper (.linear.weight)"
    NH, HD, E = arch['num_heads'], arch['head_dim'], arch['embed_dim']
    keep  = torch.where(head_mask)[0]
    nk    = len(keep)
    new_E = nk * HD

    def prune_qkv(proj):
        W = proj.linear.weight.data.view(NH, HD, E)
        proj.linear.weight = nn.Parameter(W[keep].reshape(nk*HD, E).contiguous())
        proj.linear.out_features = new_E
        if proj.linear.bias is not None:
            proj.linear.bias = nn.Parameter(proj.linear.bias.data.view(NH,HD)[keep].reshape(-1).contiguous())

    def prune_o(proj):
        W = proj.linear.weight.data.view(E, NH, HD)
        proj.linear.weight = nn.Parameter(W[:, keep, :].reshape(E, nk*HD).contiguous())
        proj.linear.in_features = new_E

    prune_qkv(attn.q_proj)
    prune_qkv(attn.k_proj)
    prune_qkv(attn.v_proj)
    prune_o(attn.o_proj)
    attn.num_heads = nk
    attn.num_key_value_heads = nk
    return nk

def prune_mlp(mlp, arch, ffn_score):
    "Prune MoshiGatingMLP — SwiGLU gate+value paired pruning"
    half = arch['ffn_half']
    gate_s = ffn_score[:half].float()
    val_s  = ffn_score[half:].float()
    pair_s = (gate_s + val_s) / 2.0
    nk   = max(64, int(half * (1 - CONFIG['ffn_pruning_ratio'])))
    keep = torch.argsort(pair_s, descending=True)[:nk].sort().values

    W1 = mlp.fc1.weight.data
    mlp.fc1.weight = nn.Parameter(
        torch.cat([W1[:half][keep], W1[half:][keep]], dim=0).contiguous()
    )
    mlp.fc1.out_features = 2 * nk
    if mlp.fc1.bias is not None:
        b = mlp.fc1.bias.data
        mlp.fc1.bias = nn.Parameter(torch.cat([b[:half][keep], b[half:][keep]]).contiguous())

    mlp.fc2.weight = nn.Parameter(mlp.fc2.weight.data[:, keep].contiguous())
    mlp.fc2.in_features = nk
    return nk

def head_mask_fn(scores, ratio):
    nk   = max(1, int(len(scores) * (1 - ratio)))
    keep = torch.argsort(scores, descending=True)[:nk]
    m    = torch.zeros(len(scores), dtype=torch.bool)
    m[keep] = True
    return m

# ══════════════════════════════════════════
# 5.3 BUILD STUDENT — Deep copy + prune
# ══════════════════════════════════════════
print("\n🔄 Creating student (deep copy)...")
student_lm = copy.deepcopy(teacher_lm)
student_lm.train()
for p in student_lm.parameters():
    p.requires_grad_(True)

student_tf, _ = find_temporal_transformer(student_lm)

# Apply depth pruning — rebuild layers list
print("✂️  Applying depth pruning...")
new_layers = nn.ModuleList([student_tf.layers[i] for i in keep_indices])
student_tf.layers = new_layers

# Update ARCH for student (fewer layers now)
ARCH_STUDENT = ARCH.copy()
ARCH_STUDENT['num_layers'] = n_keep

# Apply width pruning on kept layers
print("✂️  Applying width pruning...")
head_masks = {
    new_l: head_mask_fn(head_scores_final[orig_l], CONFIG['head_pruning_ratio'])
    for new_l, orig_l in enumerate(keep_indices)
}

print(f"   {'Layer':>5} | {'Heads':>10} | {'FFN pairs':>12}")
print(f"   {'-'*5}-+-{'-'*10}-+-{'-'*12}")
for new_l, orig_l in enumerate(keep_indices):
    layer = student_tf.layers[new_l]
    nh = prune_attention(layer.self_attn, ARCH, head_masks[new_l])
    nf = prune_mlp(layer.mlp, ARCH, ffn_scores_final[orig_l])
    if new_l % 4 == 0:
        print(f"   {new_l:>5} | {nh:>4}/{ARCH['num_heads']:<4} | {nf:>5}/{ARCH['ffn_half']:<5}")

t_p = sum(p.numel() for p in teacher_lm.parameters()) / 1e9
s_p = sum(p.numel() for p in student_lm.parameters()) / 1e9
cmp = (1 - s_p/t_p) * 100

print(f"\n{'='*50}")
print(f"  PRUNING COMPLETE")
print(f"{'='*50}")
print(f"  Teacher : {t_p:.2f}B params")
print(f"  Student : {s_p:.2f}B params")
print(f"  Reduction: {cmp:.1f}%")
print(f"  Layers  : {ARCH['num_layers']} → {n_keep}")
print(f"{'='*50}")

torch.cuda.empty_cache()
gc.collect()


## 📚 Section 6: Dataset

In [ ]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

def tokenize(text, max_len):
    if tokenizer:
        enc = tokenizer(text, max_length=max_len, truncation=True,
                        padding='max_length', return_tensors='pt')
        return enc['input_ids'].squeeze(0)
    return torch.randint(100, 32000, (max_len,))

class DistillDataset(Dataset):
    def __init__(self, texts, max_len):
        self.samples = []
        for t in texts:
            ids    = tokenize(t, max_len)
            labels = ids.clone()
            labels[:-1] = ids[1:]
            labels[-1]  = -100
            self.samples.append({'input_ids': ids, 'labels': labels})

    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]

print("🔄 Collecting training data...")
texts = []

# PersonaPlex-style prompts
pp = [
    "You are a wise and friendly teacher. Answer questions or provide advice.",
    "You enjoy having a good conversation.",
    "You enjoy having a good conversation. Have a casual discussion about eating at home versus dining out.",
    "You work for CitySan Services which is a waste management company. Your name is Ayelen Lucero.",
    "You work for Jerusalem Shakshuka which is a restaurant. Your name is Owen Foster.",
    "You enjoy having a good conversation. Have a reflective conversation about career changes.",
    "You are a helpful medical assistant. Answer health questions clearly.",
    "You are a travel agent helping customers plan their vacation.",
    "You enjoy discussing philosophy and the meaning of life.",
    "You are a friendly chef sharing cooking tips and recipes.",
] * 300
texts.extend(pp)
print(f"   PP prompts  : {len(pp)}")

try:
    c4 = load_dataset('allenai/c4', 'en', split='train', streaming=True)
    for i, row in enumerate(c4):
        if i >= 5000: break
        texts.append(row['text'][:600])
    print(f"   C4 samples  : 5000")
except Exception as e:
    print(f"   C4 failed: {e}")

try:
    ls = load_dataset("openslr/librispeech_asr", "clean", split="train.360", streaming=True)
    for i, row in enumerate(ls):
        if i >= 5000: break
        texts.append(row['text'])
    print(f"   LibriSpeech : 5000")
except Exception as e:
    print(f"   LibriSpeech failed: {e}")

print(f"   Total       : {len(texts)}")

print("\n🔄 Tokenizing...")
t0 = time.time()
ds = DistillDataset(texts, CONFIG['max_seq_len'])
loader = DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=True,
                    pin_memory=True, num_workers=2, drop_last=True)

steps_per_epoch = len(loader) // CONFIG['grad_accum_steps']
total_steps     = steps_per_epoch * CONFIG['distill_epochs']
print(f"\n✅ Dataset ready in {time.time()-t0:.0f}s")
print(f"   Samples       : {len(ds)}")
print(f"   Steps/epoch   : {steps_per_epoch}")
print(f"   Total steps   : {total_steps}")


## ⚙️ Section 7: Optimizer + Auto-Resume

In [ ]:
import bitsandbytes as bnb

def find_latest_ckpt(drive_dir):
    "Find latest epoch checkpoint on Drive"
    ckpts = glob.glob(os.path.join(drive_dir, "ckpt_epoch*.pt"))
    if not ckpts:
        return None, 0
    def get_epoch(p):
        try: return int(os.path.basename(p).split('epoch')[1].split('_')[0])
        except: return 0
    ckpts.sort(key=get_epoch)
    return ckpts[-1], get_epoch(ckpts[-1])

# ── Check for existing checkpoint ──
latest_ckpt, start_epoch = find_latest_ckpt(CONFIG['drive_dir'])

if latest_ckpt:
    print(f"🔄 RESUMING from: {os.path.basename(latest_ckpt)}")
    ckpt = torch.load(latest_ckpt, map_location=CONFIG['device'])
    student_lm.load_state_dict(ckpt['model_state_dict'])
    start_epoch_num = ckpt['epoch']
    best_loss       = ckpt['loss']
    print(f"   Resumed from epoch {start_epoch_num}, loss={best_loss:.4f}")
else:
    start_epoch_num = 0
    best_loss       = float('inf')
    print("🆕 Fresh start — no checkpoint found")

end_epoch  = CONFIG['distill_epochs']
remaining  = end_epoch - start_epoch_num
steps_left = (len(loader) // CONFIG['grad_accum_steps']) * remaining

# ── 8-bit AdamW — NVIDIA lr=1e-4, min=1e-5 ──
optimizer = bnb.optim.AdamW8bit(
    student_lm.parameters(),
    lr=CONFIG['distill_lr'],
    weight_decay=0.01,
    eps=1e-8,
)

# ── Cosine decay with linear warmup (NVIDIA Minitron) ──
total_steps  = (len(loader) // CONFIG['grad_accum_steps']) * end_epoch
warmup_steps = CONFIG['warmup_steps']  # NVIDIA: 40 steps
min_lr_ratio = CONFIG['min_lr'] / CONFIG['distill_lr']

def lr_schedule(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
    return max(min_lr_ratio, cosine)

from torch.optim.lr_scheduler import LambdaLR
scheduler = LambdaLR(optimizer, lr_schedule)

# Move student to GPU
student_lm = student_lm.to(CONFIG['device'])
student_lm.gradient_checkpointing_enable()

# Teacher to CPU — save VRAM for optimizer states
teacher_lm = teacher_lm.cpu()

vram = torch.cuda.memory_allocated() / 1e9
vram_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
print(f"\n✅ Ready")
print(f"   Student GPU   : {s_p:.2f}B")
print(f"   Teacher CPU   : {t_p:.2f}B (offloaded)")
print(f"   VRAM used     : {vram:.1f} GB")
print(f"   VRAM free     : {vram_free:.1f} GB")
print(f"   Start epoch   : {start_epoch_num+1}")
print(f"   End epoch     : {end_epoch}")
print(f"   Steps/epoch   : {len(loader)//CONFIG['grad_accum_steps']}")
print(f"   LR peak       : {CONFIG['distill_lr']} → min {CONFIG['min_lr']}")


## 🎓 Section 8: Distillation Training

**Loss function (NVIDIA Minitron style):**
```
L = α_ce × CE  +  α_kd × KL  +  α_hidden × MSE_hidden
```
Teacher on **CPU** during student backward — GPU free for optimizer states.
Teacher moved to GPU only for forward pass → back to CPU.


In [ ]:
import time, os, glob, shutil
from datetime import datetime
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets

# ── Live plot setup ──
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('PersonaPlex Distillation — Live', fontsize=13, fontweight='bold')
plot_data = {'total': [], 'ce': [], 'kd': [], 'hid': [], 'steps': []}
plt.tight_layout()

def update_plot():
    if len(plot_data['steps']) < 2:
        return
    for ax, (key, color, title) in zip(axes, [
        ('total','navy','Total Loss'),
        ('ce','darkorange','CE Loss'),
        ('kd','forestgreen','KL Loss'),
        ('hid','crimson','Hidden MSE'),
    ]):
        ax.clear()
        ax.plot(plot_data['steps'], plot_data[key], color=color, lw=1.5)
        if len(plot_data[key]) > 10:
            k = min(10, len(plot_data[key])//3)
            sm = np.convolve(plot_data[key], np.ones(k)/k, mode='valid')
            ax.plot(plot_data['steps'][k-1:], sm, color=color, lw=2.5, alpha=0.5, ls='--')
        ax.set_title(title, fontweight='bold', fontsize=10)
        ax.set_xlabel('Step')
        ax.grid(alpha=0.3)
    plt.tight_layout()
    clear_output(wait=True)
    display(fig)

# ── Training ──
steps_per_epoch = len(loader) // CONFIG['grad_accum_steps']
history = {'epoch_loss': [], 'epoch_ce': [], 'epoch_kd': [], 'epoch_hid': []}
global_step = 0
train_start = time.time()
tokens_total = 0

print("="*65)
print(f"  DISTILLATION: Epoch {start_epoch_num+1} → {end_epoch}")
print("="*65)
print(f"  Teacher : CPU offloaded | Student : GPU")
print(f"  Loss    : {CONFIG['alpha_ce']} CE + {CONFIG['alpha_kd']} KD + {CONFIG['alpha_hidden']} MSE")
print("="*65)

teacher_lm.eval()

for epoch in range(start_epoch_num, end_epoch):
    student_lm.train()
    e_loss = e_ce = e_kd = e_hid = 0.0
    n_valid = n_skip = 0
    t_fwd_s = t_bwd_s = 0.0
    epoch_start = time.time()
    epoch_tokens = 0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        ids    = batch['input_ids'].to(CONFIG['device'])
        labels = batch['labels'].to(CONFIG['device'])
        B, T   = ids.shape
        epoch_tokens += B * T

        aud_u = torch.randint(0, CONFIG['audio_vocab_size'],
                              (B, CONFIG['num_audio_codebooks'], T),
                              dtype=torch.long, device=CONFIG['device'])
        aud_a = torch.randint(0, CONFIG['audio_vocab_size'],
                              (B, CONFIG['num_audio_codebooks'], T),
                              dtype=torch.long, device=CONFIG['device'])

        # Student forward — GPU
        t0 = time.time()
        try:
            s_out = student_lm(input_ids=ids, moshi_audio_codes=aud_a, user_audio_codes=aud_u)
        except Exception:
            n_skip += 1
            continue
        t_fwd_s += time.time() - t0

        # Teacher forward — move to GPU, forward, back to CPU
        with torch.no_grad():
            try:
                teacher_lm.cuda()
                t_out = teacher_lm(
                    input_ids=ids,
                    moshi_audio_codes=aud_a,
                    user_audio_codes=aud_u,
                )
                t_lg  = t_out.logits[:, :-1].float().detach()
                t_hid = t_out.last_hidden_state.float().detach()
                del t_out
                teacher_lm.cpu()
                torch.cuda.empty_cache()
            except Exception:
                teacher_lm.cpu()
                n_skip += 1
                continue

        if s_out.logits is None:
            n_skip += 1
            continue

        s_lg  = s_out.logits[:, :-1].float()
        s_hid = s_out.last_hidden_state.float()

        # Losses
        l_ce = F.cross_entropy(
            s_lg.reshape(-1, s_lg.size(-1)),
            labels[:, 1:].reshape(-1),
            ignore_index=-100
        )
        T_t = CONFIG['temperature']
        l_kd = F.kl_div(
            F.log_softmax(s_lg / T_t, dim=-1),
            F.softmax(t_lg  / T_t, dim=-1),
            reduction='batchmean'
        )
        l_hid = F.mse_loss(s_hid, t_hid)

        loss = CONFIG['alpha_ce']*l_ce + CONFIG['alpha_kd']*l_kd + CONFIG['alpha_hidden']*l_hid

        t0 = time.time()
        (loss / CONFIG['grad_accum_steps']).backward()
        t_bwd_s += time.time() - t0

        e_loss += loss.item()
        e_ce   += l_ce.item()
        e_kd   += l_kd.item()
        e_hid  += l_hid.item()
        n_valid += 1
        tokens_total += B * T

        del t_lg, t_hid, s_lg, s_hid
        if step % 50 == 0:
            torch.cuda.empty_cache()

        if (step + 1) % CONFIG['grad_accum_steps'] == 0:
            nn.utils.clip_grad_norm_(student_lm.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

            avg    = e_loss  / max(n_valid, 1)
            a_ce   = e_ce    / max(n_valid, 1)
            a_kd   = e_kd    / max(n_valid, 1)
            a_hid  = e_hid   / max(n_valid, 1)
            cur_lr = scheduler.get_last_lr()[0]

            # Tokens/sec
            elapsed   = time.time() - epoch_start
            tok_per_s = epoch_tokens / max(elapsed, 1)
            vram      = torch.cuda.memory_allocated() / 1e9
            vram_free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9

            # Steps done this epoch
            steps_done = global_step % steps_per_epoch or steps_per_epoch
            eta_epoch  = (elapsed / steps_done) * (steps_per_epoch - steps_done)
            eta_total  = (time.time()-train_start) / max(global_step,1) * (steps_per_epoch*end_epoch - global_step)

            plot_data['steps'].append(global_step)
            plot_data['total'].append(avg)
            plot_data['ce'].append(a_ce)
            plot_data['kd'].append(a_kd)
            plot_data['hid'].append(a_hid)

            # Print every 10 steps
            if global_step % 10 == 0:
                print(f"  Ep{epoch+1:02d}|step{global_step:04d}/{steps_per_epoch*end_epoch} | loss={avg:.4f} ce={a_ce:.4f} kd={a_kd:.2f} hid={a_hid:.4f} | lr={cur_lr:.1e} | {tok_per_s:.0f}tok/s | vram={vram:.1f}/{vram+vram_free:.1f}GB | eta={eta_epoch/60:.1f}m")

            # Live plot every 50 steps
            if global_step % 50 == 0:
                try:
                    update_plot()
                except:
                    pass

    # Epoch complete
    epoch_time  = time.time() - epoch_start
    total_time  = time.time() - train_start
    avg_loss    = e_loss / max(len(loader), 1)
    tok_per_s   = epoch_tokens / max(epoch_time, 1)
    vram        = torch.cuda.memory_allocated() / 1e9
    epochs_done = epoch - start_epoch_num + 1
    epochs_left = end_epoch - epoch - 1
    eta         = (total_time / epochs_done) * epochs_left if epochs_done > 0 else 0

    history['epoch_loss'].append(avg_loss)
    history['epoch_ce'].append(e_ce / max(len(loader),1))
    history['epoch_kd'].append(e_kd / max(len(loader),1))
    history['epoch_hid'].append(e_hid / max(len(loader),1))

    print(f"\n{'='*65}")
    print(f"  EPOCH {epoch+1:02d}/{end_epoch} COMPLETE")
    print(f"  loss={avg_loss:.4f} | ce={e_ce/max(len(loader),1):.4f} | kd={e_kd/max(len(loader),1):.3f} | hid={e_hid/max(len(loader),1):.4f}")
    print(f"  tok/s={tok_per_s:.0f} | vram={vram:.1f}GB | time={epoch_time/60:.1f}min | eta={eta/3600:.2f}hr")
    print(f"{'='*65}\n")

    # Save checkpoint
    timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
    ckpt_name  = f"ckpt_epoch{epoch+1:02d}_loss{avg_loss:.4f}_{timestamp}.pt"
    local_path = os.path.join(CONFIG['local_dir'], ckpt_name)
    drive_path = os.path.join(CONFIG['drive_dir'], ckpt_name)

    torch.save({
        'epoch'            : epoch + 1,
        'loss'             : avg_loss,
        'model_state_dict' : student_lm.state_dict(),
        'metadata'         : {'global_step': global_step, 'arch': ARCH_STUDENT}
    }, local_path)

    if DRIVE_OK:
        shutil.copy2(local_path, drive_path)
        os.remove(local_path)
        print(f"  💾 Saved to Drive: {ckpt_name}")

        # Delete old checkpoints — keep only latest
        all_ckpts = sorted(glob.glob(os.path.join(CONFIG['drive_dir'], "ckpt_epoch*.pt")))
        for old in all_ckpts:
            if old != drive_path:
                os.remove(old)
                print(f"  🗑️  Deleted: {os.path.basename(old)}")

        # Best checkpoint
        if avg_loss < best_loss:
            best_loss = avg_loss
            best_path = os.path.join(CONFIG['drive_dir'], 'best_checkpoint.pt')
            shutil.copy2(drive_path, best_path)
            print(f"  🏆 New best! loss={avg_loss:.4f}")
    else:
        print(f"  💾 Saved locally: {local_path}")

# Final plot
try:
    update_plot()
except:
    pass

total_time = time.time() - train_start
print(f"\n✅ Training complete!")
print(f"   Total time : {total_time/3600:.2f} hr")
print(f"   Best loss  : {best_loss:.4f}")


## 💾 Section 9: Final Save

In [ ]:
# Load best checkpoint
best_path = os.path.join(CONFIG['drive_dir'], 'best_checkpoint.pt')
if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location=CONFIG['device'])
    student_lm.load_state_dict(ckpt['model_state_dict'])
    print(f"✅ Best checkpoint loaded (epoch {ckpt['epoch']}, loss={ckpt['loss']:.4f})")

student_lm.eval()

# Save HuggingFace format
save_path = os.path.join(CONFIG['drive_dir'], 'personaplex_slim_final')
os.makedirs(save_path, exist_ok=True)

try:
    student_lm.save_pretrained(save_path, safe_serialization=True)
    print(f"✅ HF format saved: {save_path}")
except Exception as e:
    torch.save(student_lm.state_dict(), os.path.join(save_path, 'weights.pt'))
    print(f"✅ State dict saved: {save_path}")

# Metadata
meta = {
    'base_model'         : CONFIG['model_id'],
    'teacher_params_B'   : t_p,
    'student_params_B'   : s_p,
    'compression_pct'    : (1-s_p/t_p)*100,
    'depth_ratio'        : CONFIG['depth_pruning_ratio'],
    'head_ratio'         : CONFIG['head_pruning_ratio'],
    'ffn_ratio'          : CONFIG['ffn_pruning_ratio'],
    'layers_kept'        : n_keep,
    'layers_removed'     : n_remove,
    'best_loss'          : best_loss,
}
with open(os.path.join(save_path, 'metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

s_final = sum(p.numel() for p in student_lm.parameters()) / 1e9
print(f"""
╔══════════════════════════════════════════════════╗
║   PERSONAPLEX SLIM V8 — FINAL SUMMARY            ║
╠══════════════════════════════════════════════════╣
║  Teacher       : {t_p:.2f}B params                   ║
║  Student       : {s_final:.2f}B params                   ║
║  Compression   : {(1-s_final/t_p)*100:.1f}%                          ║
╠══════════════════════════════════════════════════╣
║  Depth pruning : {CONFIG['depth_pruning_ratio']*100:.0f}% ({ARCH['num_layers']}→{n_keep} layers)             ║
║  Head pruning  : {CONFIG['head_pruning_ratio']*100:.0f}% per layer                ║
║  FFN pruning   : {CONFIG['ffn_pruning_ratio']*100:.0f}% per layer                ║
║  Best loss     : {best_loss:.4f}                      ║
╚══════════════════════════════════════════════════╝
""")
